# Benchmarking the Tuna devices performance using common measurement routines

**Author:** Marios Samiotis (m.samiotis@tudelft.nl)

**Date:** April 29, 2026

**Last updated:** November 7, 2026

# 1. Introduction

In this notebook we will use some of the most common benchmark measurement routines to assess the performance of any of the Tuna superconducting quantum backends which exist at the [Quantum Inspire 2.0](https://www.quantum-inspire.com/) cloud services.

These are the same routines that are used in a superconducting qubits lab environment in order to characterize a quantum processor: we will measure the qubits relaxation and dephasing times, check the quality of the single-qubit gates, and benchmark the two-qubit CZ gate and the preparation of entangled states. All of them are executed with a single line of code through the "DeviceControl" class of this repository.

Prior to using this notebook, for optimal performance, make sure that you are running it within a customized Python 3.12 environment which includes the packages "quantuminspire" and "qiskit-quantuminspire".

For details on how to create such a Python environment, follow the installation instructions in the [README file](https://github.com/DiCarloLab-Delft/QuantumInspireUtilities/blob/main/README.md).

If you are a beginner, we highly recommend going through the Getting Started guide for the [Tuna backends](https://github.com/DiCarloLab-Delft/QuantumInspireUtilities/blob/main/qi_utilities/notebook_guides/0.0_getting_started_tuna_devices.ipynb) before continuing with this notebook, since the concepts of mid-circuit measurements and readout error mitigation are used extensively in all measurement routines below.

Useful links:
1. [Tuna Backends Operational Specifics](https://www.quantum-inspire.com/kbase/tuna-operational-specifics/)
2. [Backend Performance Dashboards](https://dashboard.quantum-inspire.com/) [live updates]
3. [Join our Slack community!](https://join.slack.com/t/qisuperconducting/shared_invite/zt-35o7zitdh-_9QPmB53hhLy12Eat5gwWA)

Further reading on the measurement routines used in this notebook:
1. [Entanglement and Quantum Error Correction with Superconducting Qubits](https://arxiv.org/abs/1311.6759), Matthew Reed, PhD Thesis (2013), Chapter 5, Section 5.2.
2. [Control for Programmable Superconducting Quantum Systems](https://doi.org/10.4233/uuid:0a2ba212-f6bf-4c64-8f3d-b707f1e44953), Michiel Adriaan Rol, PhD Thesis (2020).

First, we run the following cell in order to login to the Quantum Inspire platform. You need an account in order to login to the platform.

Please click on "Confirm" in the pop-up browser window to login.

In [ ]:
! qi login "https://api.quantum-inspire.com"

Run the following cell for all necessary library imports,

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_quantuminspire.qi_provider import QIProvider
from qi_utilities.device_simulation.simulators import NoisySimulator # use the simulator in case all backends are unavailable!
from qi_utilities.measurements.device_control import DeviceControl
from qi_utilities.measurements.measurement_classes import (RabiMeasurement, T1_Measurement, FlippingMeasurement,
                                                           AllXYMeasurement, ConditionalOscMeasurement, BellStateMeasurement)

# 2. Connecting to the backend

We first connect to the Quantum Inspire provider, by running the following cell,

In [ ]:
provider = QIProvider()

The following command prints the full list of all available backends of the platform,

In [ ]:
provider.backends()

We now connect to a Tuna backend by running the following cell,

In [ ]:
backend_name = "Tuna-17" # check availability!
backend = provider.get_backend(name=backend_name)

In case all backends are unavailable, one may instead use a noisy simulator of the device by running the following cell,

In [ ]:
backend = NoisySimulator("Starmon-7")

By running the cell below, the connectivity of the processor is shown (otherwise known as "coupling map"),

In [ ]:
backend.coupling_map.draw()

Any single- or multi-qubit gates that may be applied on the qubits of the Tuna backends are referred to as "instructions". The full list of allowed instructions of the Tuna backend is shown by running the following cell,

In [ ]:
backend.instructions

When executing a quantum algorithm on any backend, one needs to choose the number of measurement shots to be performed for the given quantum circuit. The maximum number of allowed shots on the Tuna backend can be obtained by running the command

In [ ]:
backend.max_shots

# 3. The DeviceControl class

All measurement routines of this notebook are executed through the "DeviceControl" class, which we initialize by passing the backend,

In [ ]:
device = DeviceControl(backend) # optionally, pass directory = "..." to specify where the project records are stored

Each measurement routine is executed by calling the corresponding "measure_" method of the class. Behind the scenes, every routine:

1. creates a single quantum circuit containing one measurement block per measurement point, using mid-circuit measurements (see Section 4 of the [Getting Started guide](https://github.com/DiCarloLab-Delft/QuantumInspireUtilities/blob/main/qi_utilities/notebook_guides/0.0_getting_started_tuna_devices.ipynb)),
2. appends at the end of the circuit the readout calibration circuits, used for mitigating readout errors (see Section 3.2 of the [Getting Started guide](https://github.com/DiCarloLab-Delft/QuantumInspireUtilities/blob/main/qi_utilities/notebook_guides/0.0_getting_started_tuna_devices.ipynb)),
3. executes the job on the backend, and stores the project record locally inside the 'Documents/QuantumInspireProjects' folder,
4. analyzes the data, and stores the plots (PNG format) and the processed data (JSON format) inside the project folder,
5. prints the extracted values, which are also stored as attributes of the class.

All qubits in the input "qubit_list" are measured simultaneously. The input qubit_list can either be a list of qubit indices, e.g. qubit_list = [0, 2] for qubits Q0 and Q2, or 'all' for all qubits of the backend.

Since we are using multiple measurement blocks within a single circuit, we have to be careful with the number of shots: if we request too many shots, we risk having the job fail due to memory limitations of the readout instruments. By default, all routines use the following number of shots,

In [ ]:
device.num_shots

which can be overwritten for every measurement through the input "num_shots".

# 4. Single-qubit benchmark measurements

In this section we perform the most common single-qubit benchmark measurements. The order of the measurements below follows the order in which those are usually performed in the lab when tuning up a qubit: we first check the qubit rotations (Rabi), then measure the coherence times ($T_1$, $T_2^*$ and $T_2^{\text{echo}}$), and finally check the single-qubit gates more precisely (flipping, AllXY).

One is advised to consult the appropriate Tuna backend [performance dashboard](https://dashboard.quantum-inspire.com/) when choosing the measurement parameters below, such as the total measurement times, which depend on the coherence times of the qubits at a given moment in time.

## 4.1. Rabi oscillation

In a [Rabi oscillation](https://en.wikipedia.org/wiki/Rabi_cycle) the qubit is rotated around the x (or y) axis of the Bloch sphere by a varied rotation angle, and then measured. Since all single-qubit gates on the Tuna backends have a fixed duration, a varied rotation angle translates to a varied amplitude of the applied microwave pulse. The excited state population $P(|1\rangle)$ has a sinusoidal response to the rotation angle, which is fitted with a cosine function.

This experiment serves as a general check of whether the qubit behaves well as a two-level system: a distorted oscillation, or a $2\pi$ rotation that does not return the qubit to its ground state, indicates leakage outside of the computational space (see [Reed's thesis](https://arxiv.org/abs/1311.6759), Section 5.2.1).

The printed value is the fitted excited state population after a $\pi$ rotation, which ideally is equal to 1.

In [ ]:
device.measure_rabi(qubit_list='all', # e.g. qubit_list=[0, 5, 6]
                    rotation_angles = np.linspace(-np.pi, np.pi, num=51),
                    rotation_axis='x',
                    num_shots = 2**12,
                    use_ro_cal_points=True)

The latest executed quantum circuit can always be visualized by running the following cell,

In [ ]:
device.latest_qc.draw('mpl')

## 4.2. Relaxation time $T_1$

In a $T_1$ measurement the qubit is brought to its excited state $|1\rangle$ with a $\pi$ pulse, and is then left idle for a varied time $t$ before being measured. Due to energy relaxation, the excited state population decays exponentially to the ground state, as $P(|1\rangle) = a\cdot\exp(-t/T_1) + b$, from which the relaxation time $T_1$ is extracted. For more information, refer to [A Quantum Engineer's Guide to Superconducting Qubits](https://arxiv.org/abs/1904.06560v5), Section III. B. 2.

This is the same experiment that we performed manually in Section 4 of the Getting Started guide. The total measurement time should be chosen approximately 3 to 5 times the expected $T_1$ of the qubits, for a reliable fit. Notice that the idle times are applied in multiples of the 20 ns cycle time of the Tuna backends.

The printed values are the $T_1$ times in units of $\mu s$.

In [ ]:
device.measure_T1(qubit_list='all', # e.g. qubit_list=[0, 5, 6]
                  measurement_times=np.linspace(0, 50e-6, num=41), # in units of [s]
                  num_shots=2**12)

## 4.3. Dephasing time $T_2^*$ (Ramsey experiment)

In a Ramsey experiment the qubit is brought to the equator of the Bloch sphere with a $\pi/2$ pulse, is left to evolve freely for a varied time $t$, and is then rotated back with a second $\pi/2$ pulse prior to measurement. During the free evolution, any detuning of the qubit from its drive frequency acts as a rotation around the z axis, while dephasing decays the coherence of the qubit. The excited state population thus follows a damped oscillation, from which the dephasing time $T_2^*$ is extracted.

In order to clearly distinguish the decay from the oscillation, an artificial detuning is added by rotating the phase of the second $\pi/2$ pulse, such that five oscillations occur within the total measurement time (see [Reed's thesis](https://arxiv.org/abs/1311.6759), Section 5.2.2).

In case a double beating is detected in the data (two oscillation frequencies, e.g. due to a qubit frequency switching between two values), the measurement is automatically re-launched with two adjusted artificial detunings, so do not be surprised if a second job is uploaded!

Notice that the idle times are applied in multiples of the 20 ns cycle time of the Tuna backends, therefore the measurement times should be chosen as multiples of 20 ns.

The printed values are the $T_2^*$ times in units of $\mu s$.

In [ ]:
device.measure_T2_ramsey(qubit_list='all', # e.g. qubit_list=[0, 8]
                         measurement_times=np.linspace(0, 1.2e-6, num=61), # in units of [s], steps of 20 ns
                         num_shots=2**12)

## 4.4. Dephasing time $T_2^{\text{echo}}$ (Hahn echo experiment)

The Hahn echo experiment is identical to the Ramsey experiment, with an additional $\pi$ pulse applied exactly in the middle of the free evolution time. The $\pi$ pulse reverses the direction of the qubit evolution, making the qubit insensitive to low-frequency noise. Since the noise of superconducting qubits is usually dominated by low frequencies, typically $T_2^{\text{echo}} > T_2^*$ (see [Reed's thesis](https://arxiv.org/abs/1311.6759), Section 5.2.2).

Since each half of the free evolution time is applied in multiples of the 20 ns cycle time, the measurement times should be chosen as multiples of 40 ns.

The printed values are the $T_2^{\text{echo}}$ times in units of $\mu s$.

In [ ]:
device.measure_T2_echo(qubit_list='all', # e.g. qubit_list=[0, 8]
                       measurement_times=np.linspace(0, 2.4e-6, num=61), # in units of [s], steps of 40 ns
                       num_shots=2**12)

## 4.5. Flipping experiment

The flipping experiment is used for the fine calibration of the amplitude of the $\pi$ pulses. The qubit is first brought to the equator of the Bloch sphere with a $\pi/2$ pulse (for equator=True), followed by an even number $N$ of $\pi$ pulses. Ideally the qubit remains on the equator for any $N$, with $P(|1\rangle) = 0.5$. Any small error in the pulse amplitude accumulates with every additional pulse, amplifying the error, which is then visible as a drift of $P(|1\rangle)$ away from 0.5 (see [Reed's thesis](https://arxiv.org/abs/1311.6759), Section 5.2.4).

The data are fitted with both a line and a cosine, and the best fit is chosen. The printed "scale factors" are the factors by which the pulse amplitude would need to be multiplied in order to correct for the amplitude error, with ideal value 1 (same as in [PycQED](https://github.com/DiCarloLab-Delft/PycQED_py3)).

In [ ]:
device.measure_flipping(qubit_list='all',
                        num_shots=2**12,
                        max_number_of_flips=30, # must be an even number
                        equator=True)

## 4.6. AllXY experiment

The AllXY experiment is a quick and sensitive test of the single-qubit gates. All 21 combinations of two consecutive rotations, taken from the identity and rotations around the x and y axes by an angle of $\pi/2$ or $\pi$, are applied on the qubit prior to measurement, each combination measured twice. Ideally, the qubit ends up in the ground state for the first 5 combinations, on the equator for the next 12 combinations and in the excited state for the last 4 combinations, giving a characteristic "staircase" pattern. Different gate errors, such as amplitude errors and detuning, produce distinct deviations from the ideal staircase (see [Reed's thesis](https://arxiv.org/abs/1311.6759), Section 5.2.3).

The printed values are the mean absolute deviations from the ideal staircase, with ideal value 0 (see [Rol's thesis](https://doi.org/10.4233/uuid:0a2ba212-f6bf-4c64-8f3d-b707f1e44953), Section 5.2.3).

In [ ]:
device.measure_allxy(qubit_list='all', # e.g. qubit_list=[0, 8]
                     num_shots=2**12)

# 5. Two-qubit benchmark measurements

In this section we benchmark the two-qubit CZ gate and the preparation of entangled states. The two-qubit measurement routines take as input a list of qubit pairs, "qubit_pairs", where the qubits of each pair need to be directly coupled to each other (see the coupling map in Section 2). Multiple qubit pairs can be measured simultaneously, as long as a single qubit is not used in multiple pairs.

In the examples below we use the qubit pair Q0 - Q2; one is advised to consult the [performance dashboard](https://dashboard.quantum-inspire.com/) for choosing the most performant pairs at a given moment in time.

## 5.1. Conditional oscillation experiment

The conditional oscillation experiment is used for measuring the conditional phase of the CZ gate and for estimating its leakage (see [Rol's thesis](https://doi.org/10.4233/uuid:0a2ba212-f6bf-4c64-8f3d-b707f1e44953), Section 7.5.4). For each qubit pair [q_targ, q_ctrl], two variants of a Ramsey-like experiment are performed:

1. "Off": the target qubit is brought to the equator with a $\pi/2$ pulse, while the control qubit remains in its ground state. The CZ gate is applied, followed by a final $\pi/2$ pulse on the target qubit, whose phase is varied.
2. "On": same as above, but the control qubit is brought to its excited state with a $\pi$ pulse before the CZ gate, and back to its ground state with a second $\pi$ pulse before the measurement.

The target qubit oscillates as a function of the phase in both variants, and the phase difference between the two oscillations is the conditional phase of the CZ gate, which ideally is $180^\circ$. The "missing fraction" of the control qubit, which is the difference of its average excited state population between the "On" and "Off" variants, is an estimate for the leakage of the CZ gate (ideally 0). One may use cz_repetitions > 1 to amplify any phase errors.

In [ ]:
device.measure_conditional_oscillation(qubit_pairs = [[0, 2]], # [q_targ, q_ctrl]
                                       num_angles=19,
                                       num_shots = 2**12,
                                       cz_repetitions=1)

## 5.2. Bell-state tomography

Finally, we prepare one of the four [Bell states](https://en.wikipedia.org/wiki/Bell_state) ("phi_plus", "phi_minus", "psi_plus" or "psi_minus") on each qubit pair, and reconstruct the density matrix $\rho$ of the prepared state with two-qubit state tomography. Both qubits are measured in all 9 combinations of the X, Y and Z bases, from which the expectation values of all 16 two-qubit Pauli operators are obtained (see [Reed's thesis](https://arxiv.org/abs/1311.6759), Section 7.3). A physical density matrix is then reconstructed from those expectation values (Section 7.3.1 of the same thesis).

The printed values are the fidelities $F = \langle \psi | \rho | \psi \rangle$ to the target Bell state $|\psi\rangle$, with ideal value 1. Since this experiment contains only 9 measurement blocks, we can afford a larger number of shots.

In [ ]:
device.measure_bell_state_fidelity(qubit_pairs = [[0, 2]],
                                   bell_state = "phi_plus",
                                   num_shots = 2**14)

# 6. Accessing the latest results

The latest results of each measurement routine are stored as attributes of the DeviceControl class, and can be accessed at any moment by running the following cell,

In [ ]:
print(f"Rabi amplitudes: {device.rabi_amplitudes}")
print(f"T1 values: {device.T1_values}")
print(f"T2* values: {device.T2_ramsey_values}")
print(f"T2 echo values: {device.T2_echo_values}")
print(f"Flipping parameters: {device.flipping_parameters}")
print(f"AllXY deviations: {device.allxy_deviations}")
print(f"Conditional oscillation parameters: {device.cond_osc_params}")
print(f"Bell-state fidelities: {device.bell_state_fidelities}")

# 7. Loading the data of previous measurements

The data of an already executed measurement can be loaded from the local project record by using the "load_data" method of the corresponding measurement class. One only needs to enter the timestamp of the experiment, which is shown in the title of all measurement plots, in the format "YYYYMMDD_HHMMSS". The data analysis is then performed again, and the full experiment data are returned as a dictionary.

Below, we load the data of a Rabi oscillation (enter the timestamp of your own measurement),

In [ ]:
rabi_loaded_data = RabiMeasurement.load_data(job_timestamp='20261108_150315')

and plot the excited state population of a qubit using the loaded data,

In [ ]:
experiment_name = rabi_loaded_data['Experiment name']
experiment_timestamp = rabi_loaded_data['Experiment timestamp']
backend_name = rabi_loaded_data['Backend name']
num_shots = rabi_loaded_data['Number of shots']
qubit = 'Q0'

rotation_axis = rabi_loaded_data['Rotation axis']
rotation_angles = rabi_loaded_data['Rotation angles [rad]']
prob_excited = [rabi_loaded_data['Processed data'][qubit][meas_idx]['1'] for meas_idx in range(len(rabi_loaded_data['Processed data'][qubit]))]

fig, ax = plt.subplots(figsize=(18, 5), dpi=300)
ax.plot(rotation_angles,
        prob_excited,
        label=f'Qubit {qubit}',
        alpha=0.6,
        color = 'C0',
        marker='o')
ax.set_xlabel('Applied rotation (rad)')
ax.set_ylabel(r'Population $|1\rangle$')
ax.set_title(
    f'{experiment_name}'
    f'\n{backend_name} processor | num_shots = {num_shots}'
    f'\n{experiment_timestamp}')
ax.set_ylim(-0.05, 1.05)

labels = []
for step_idx in range(len(rotation_angles)):
    angle_in_degrees = (360 / (2 * np.pi)) * rotation_angles[step_idx]
    if rotation_axis == 'x':
        step_label = f"rx{round(angle_in_degrees)}"
    if rotation_axis == 'y':
        step_label = f"ry{round(angle_in_degrees)}"
    labels.append(step_label)
label_locs = rotation_angles
ax.set_xticks(label_locs)
ax.set_xticklabels(labels, rotation=65)
ax.set_ylim(-0.05, 1.05)
ax.grid(True)

plt.legend()
plt.show()

In the same way, we can load the data of the $T_1$, flipping, AllXY, conditional oscillation and Bell-state measurements (loading is not yet available for the $T_2^*$ and $T_2^{\text{echo}}$ measurements),

In [ ]:
T1_loaded_data = T1_Measurement.load_data(job_timestamp='20261106_152350')

In [ ]:
flipping_loaded_data = FlippingMeasurement.load_data(job_timestamp='20261106_152350')

In [ ]:
allxy_loaded_data = AllXYMeasurement.load_data(job_timestamp='20261106_152350')

In [ ]:
cond_osc_loaded_data = ConditionalOscMeasurement.load_data(job_timestamp='20261106_152350')

In [ ]:
bell_state_loaded_data = BellStateMeasurement.load_data(job_timestamp='20261108_145620')

The keys of the returned dictionary show all the stored experiment data,

In [ ]:
rabi_loaded_data.keys()